# Bronze ingest — customers

Streams CSV exports of the CRM customer table out of the landing volume into the bronze `customers_raw` Delta table. The CRM dumps a fresh file every six hours; Auto Loader picks them up incrementally.

### DEGraph extraction notes

This file is the benchmark's representative for the `.ipynb` (Jupyter JSON) input format. The DEGraph extractor must (a) detect that it is JSON-formatted, (b) split cells via `nbformat`, (c) drop markdown cells, (d) drop the `%sql DESCRIBE` magic cell as a metadata query rather than a lineage operation, and (e) feed only the Python code cells to the AST visitor.

If the SQL magic were accidentally kept, the extractor would emit a spurious `Reads` edge against `main.dbdemos_ecom.customers_raw` from this file — which would falsely position the bronze ingest notebook as a consumer of its own output. The ground-truth graph asserts that this does NOT happen.

In [ ]:
%run ../_resources/setup

In [ ]:
from pyspark.sql import functions as F

## 1. Read the CRM landing zone

In [ ]:
customers_landing = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{schema_root}/customers")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(f"{volume_path}/customers")
)

## 2. Add operational metadata

In [ ]:
customers_with_meta = (
    customers_landing
        .withColumn("ingested_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
)

## 3. Write to bronze

In [ ]:
(customers_with_meta
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_root}/customers")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(f"{database}.customers_raw"))

## 4. Sanity check

Quick `DESCRIBE` of the bronze table after the stream finishes. This is a metadata query, not a lineage operation — the DEGraph extractor must drop this cell.

In [ ]:
%sql
DESCRIBE TABLE main.dbdemos_ecom.customers_raw